# Plan

- Define the constants and parameters for the analysis
- Model the system and shots etc first before anything. Write a function to create all the models, test them all and store in a pkl for any test being done with any shots etc. 
- Just get all the results in first for all of the datasets, error bars, CI etc can be found later without need to change the code initially

Things to test in the function
- Shot sweep with all models to find fidelity for both pure and mixed states
- Do this with 2 qubits and 3 qubits (can be determined from which dataset is used)
- Measure runtime for all methods (Stokes, MLE, NNs [testing runtime])

In [ ]:
# Define global constants and parameters
HIDDEN_SIZES = (256, 256, 256)
LEARNING_RATE = 5e-4
BATCH_SIZE = 64
EPOCHS = 100
DROPOUT = 0.08

In [3]:
# import all files
import os

files = sorted(os.listdir("data"))
test_files = [f for f in files if "_test" in f]
train_test_pairs = [(f.replace("_test", "_train"), f) for f in test_files]

In [ ]:
# define a function to ascertain key info from the filename
def parse_filename(filename):
    parts = filename.split("_")
    number_qubits = int(parts[0].replace("q", ""))
    state_type = parts[1]
    num_shots = int(parts[2].replace("shots", ""))
    return number_qubits, state_type, num_shots

In [ ]:
import numpy as np
# Check if a state is physical
def is_physical(rho):
    # Check if the matrix is Hermitian
    if not np.allclose(rho, rho.conj().T):
        return False
    
    # Check if the matrix is positive semi-definite
    eigenvals = np.linalg.eigvalsh(rho)
    if  np.any(eigenvals < -1e-8):
        return False
    
    # Check if the matrix is normalized
    if not np.isclose(np.trace(rho), 1.0):
        return False
    
    return True

In [ ]:
# define a function to take one train test pair and run the full training and evaluation process, returning the history and final output
import QST_core_processes as qst
import pickle
import time

def run_full_process(train_file, test_file):

    # parse the filename to get key info
    number_qubits, state_type, num_shots = parse_filename(test_file)

    results = {
        "dataset_info": train_file.replace("_train", ""),
        "qubits": number_qubits,
        "state_type": state_type,
        "shots": num_shots,
        "outputs": {}
    }
    
    # load the data
    data_train = np.load(os.path.join("data", train_file), allow_pickle=True)
    data_test = np.load(os.path.join("data", test_file), allow_pickle=True)

    def fidelities_vs_truth(pred_rhos):
        return np.array([qst.fidelity(data_test['rhos'][k], pred_rhos[k]) for k in range(len(data_test['rhos']))])
    
    # Stokes data

    start = time.time()
    rhos = qst.stokes_reconstruct_dataset(
            P = data_train['P'],
            counts = data_test['counts'],
            shots = data_test['shots'],
            n_qubits = data_train['n_qubits']
        )
    end = time.time()
    stokes_time = end - start

    results["outputs"]["Stokes"] = {
        "rhos": rhos,
        "fidelities": fidelities_vs_truth(rhos),
        "fraction_physical": np.mean([is_physical(rho) for rho in rhos]),
        "time": stokes_time
    }

    # MLE data

    start = time.time()
    rhos = qst.mle_reconstruct_dataset(
            P = data_train['P'],
            counts = data_test['counts'],
            n_qubits = data_train['n_qubits'],
            steps=200
        )
    end = time.time()
    mle_time = end - start

    results["outputs"]["MLE"] = {
        "rhos": rhos,
        "fidelities": fidelities_vs_truth(rhos),
        "fraction_physical": np.mean([is_physical(rho) for rho in rhos]),
        "time": mle_time
    }

    # Naive Data

    nn_naive = qst.NN_Builder(
        n_qubits = data_train['n_qubits'],
        model_type = "mlp",
        loss_type = "mse",
        target = "rho",
        hidden_sizes = HIDDEN_SIZES,
        dropout = DROPOUT,
        lr = LEARNING_RATE,
        batch_size = BATCH_SIZE,
        epochs = EPOCHS
    )

    _, prediction, eval_time = nn_naive.fit_and_predict(data_train, data_test)

    results["outputs"]["Naive_NN"] = {
        "rhos": prediction,
        "fidelities": fidelities_vs_truth(prediction),
        "fraction_physical": np.mean([is_physical(rho) for rho in prediction]),
        "time": eval_time
    }

    # Physics Informed Data

    nn_physics = qst.NN_Builder(
        n_qubits = data_train['n_qubits'],
        model_type = "mlp",
        loss_type = "fidelity",
        target = "tau",
        hidden_sizes = HIDDEN_SIZES,
        dropout = DROPOUT,
        lr = LEARNING_RATE,
        batch_size = BATCH_SIZE,
        epochs = EPOCHS
    )

    _, prediction, eval_time = nn_physics.fit_and_predict(data_train, data_test)

    results["outputs"]["Physics_Informed_NN"] = {
        "rhos": prediction,
        "fidelities": fidelities_vs_truth(prediction),
        "fraction_physical": np.mean([is_physical(rho) for rho in prediction]),
        "time": eval_time
    }

    # Basic CNN data

    nn_cnn = qst.NN_Builder(
        n_qubits = data_train['n_qubits'],
        model_type = "cnn",
        loss_type = "fidelity",
        target = "tau",
        hidden_sizes = HIDDEN_SIZES,
        dropout = DROPOUT,
        cnn_channels = (8, 16),
        cnn_kernel_size = 3,
        lr = LEARNING_RATE,
        batch_size = BATCH_SIZE,
        epochs = EPOCHS
    )

    _, prediction, eval_time = nn_cnn.fit_and_predict(data_train, data_test)

    results["outputs"]["CNN"] = {
        "rhos": prediction,
        "fidelities": fidelities_vs_truth(prediction),
        "fraction_physical": np.mean([is_physical(rho) for rho in prediction]),
        "time": eval_time
    }

    # CNN_overlap_mixing data

    nn_cnn_overlap = qst.NN_Builder(
        n_qubits = data_train['n_qubits'],
        model_type = "cnn",
        loss_type = "fidelity",
        target = "tau",
        hidden_sizes = HIDDEN_SIZES,
        dropout = DROPOUT,
        cnn_channels = (8, 16),
        cnn_kernel_size = 3,
        cnn_kernel_type = "proj_kernel",
        proj_kernel_metric = "overlap",
        lr = LEARNING_RATE,
        batch_size = BATCH_SIZE,
        epochs = EPOCHS,
        overlap_mixing = True
    )

    _, prediction, eval_time = nn_cnn_overlap.fit_and_predict(data_train, data_test)

    results["outputs"]["CNN_Overlap_Mixing"] = {
        "rhos": prediction,
        "fidelities": fidelities_vs_truth(prediction),
        "fraction_physical": np.mean([is_physical(rho) for rho in prediction]),
        "time": eval_time
    }

    # CNN_fidelity_mixing data

    nn_cnn_fidelity = qst.NN_Builder(
        n_qubits = data_train['n_qubits'],
        model_type = "cnn",
        loss_type = "fidelity",
        target = "tau",
        hidden_sizes = HIDDEN_SIZES,
        dropout = DROPOUT,
        cnn_channels = (8, 16),
        cnn_kernel_size = 3,
        cnn_kernel_type = "proj_kernel",
        proj_kernel_metric = "fidelity",
        lr = LEARNING_RATE,
        batch_size = BATCH_SIZE,
        epochs = EPOCHS
    )

    _, prediction, eval_time = nn_cnn_fidelity.fit_and_predict(data_train, data_test)

    results["outputs"]["CNN_Fidelity_Mixing"] = {
        "rhos": prediction,
        "fidelities": fidelities_vs_truth(prediction),
        "fraction_physical": np.mean([is_physical(rho) for rho in prediction]),
        "time": eval_time
    }

    # CNN_graph_mixing data

    nn_cnn_graph = qst.NN_Builder(
        n_qubits = data_train['n_qubits'],
        model_type = "cnn",
        loss_type = "fidelity",
        target = "tau",
        hidden_sizes = HIDDEN_SIZES,
        dropout = DROPOUT,
        cnn_channels = (8, 16),
        cnn_kernel_size = 3,
        cnn_kernel_type = "proj_graph",
        proj_kernel_metric = "fidelity",
        lr = LEARNING_RATE,
        batch_size = BATCH_SIZE,
        epochs = EPOCHS
    )

    _, prediction, eval_time = nn_cnn_graph.fit_and_predict(data_train, data_test)

    results["outputs"]["CNN_Graph_Mixing"] = {
        "rhos": prediction,
        "fidelities": fidelities_vs_truth(prediction),
        "fraction_physical": np.mean([is_physical(rho) for rho in prediction]),
        "time": eval_time
    }

    return results

In [ ]:
all_results = []
run = 1

for train_file, test_file in train_test_pairs:
    print(f"Running set {run} out of {len(train_test_pairs)}")
    results = run_full_process(train_file, test_file)
    all_results.append(results)
    run += 1

    # Save results to a file
    with open(f"all_results.pkl", "wb") as f:
        pickle.dump(all_results, f)


In [ ]:
print("ALL DONE!!")